# 13_compare_dates_full
Compares two loaded dates across the whole pipeline: pipeline runs, table growth, market state (Gold snapshot), price and discount activity, and new listings.
Parameters: start_date, end_date (start_date <= end_date, both on or after 2026-10-09).

In [0]:
from datetime import datetime
from pyspark.sql import functions as F

MIN_DATE = "2026-10-09"
dbutils.widgets.text("start_date", MIN_DATE)
dbutils.widgets.text("end_date", "")

loaded = sorted(str(r.full_date) for r in spark.sql("""
    SELECT DISTINCT d.full_date FROM workspace.gold.fact_game_daily_snapshot f
    JOIN workspace.gold.dim_date d USING (date_key)""").collect())
latest = loaded[-1]

start_date = dbutils.widgets.get("start_date").strip() or MIN_DATE
end_date = dbutils.widgets.get("end_date").strip() or latest
for d in (start_date, end_date):
    datetime.strptime(d, "%Y-%m-%d")
if start_date < MIN_DATE:
    raise ValueError(f"start_date must be on or after {MIN_DATE}")
if start_date > end_date:
    raise ValueError("start_date must be on or before end_date")
if end_date > latest:
    raise ValueError(f"end_date is after the latest loaded date ({latest})")

# a date with no data (like Oct 12) is compared using the last loaded day before it
def as_loaded(d):
    return max(x for x in loaded if x <= d)

start_eff, end_eff = as_loaded(start_date), as_loaded(end_date)
start_key, end_key = int(start_eff.replace("-", "")), int(end_eff.replace("-", ""))
missing = [str(d) for d in spark.sql(f"""
    SELECT full_date FROM workspace.gold.dim_date
    WHERE full_date BETWEEN DATE'{start_date}' AND DATE'{end_date}'""").toPandas().full_date
           if str(d) not in loaded]

print(f"comparing {start_date} (data from {start_eff}) to {end_date} (data from {end_eff})")
print("days with no data in range:", missing or "none")

comparing 2026-10-09 (data from 2026-10-09) to 2026-10-25 (data from 2026-10-25)
days with no data in range: ['2026-10-12', '2026-10-20']


In [0]:
logs = (spark.table("workspace.steam.pipeline_execution_logs")
        .withColumn("data_date", F.expr(r"try_to_date(regexp_extract(file_processed, '(\\d{4}-\\d{2}-\\d{2})', 1))"))
        .where(F.col("data_date").between(start_date, end_date))
        .where(F.col("layer") != "Pipeline-Runner")
        .withColumn("seconds", F.col("end_time").cast("double") - F.col("start_time").cast("double")))

print("Runs per layer for data dates in the range:")
display(logs.groupBy("layer").agg(
    F.count("*").alias("runs"),
    F.sum(F.when(F.col("status") == "Success", 1).otherwise(0)).alias("succeeded"),
    F.sum(F.when(F.col("status") == "Failure", 1).otherwise(0)).alias("failed"),
    F.sum("rows_inserted").alias("rows_inserted"),
    F.sum("rows_updated").alias("rows_updated"),
    F.sum("rows_quarantined").alias("rows_quarantined"),
    F.round(F.avg("seconds"), 1).alias("avg_seconds"),
    F.round(F.max("seconds"), 1).alias("max_seconds"))
  .orderBy("layer"))

Runs per layer for data dates in the range:


layer,runs,succeeded,failed,rows_inserted,rows_updated,rows_quarantined,avg_seconds,max_seconds
Bronze-to-Silver,49,49,0,246894,96095,0,14.0,27.6
Raw-to-Bronze,26,26,0,119226,7,0,32.7,47.3
Silver-to-Gold,18,18,0,1446119,1483,null,57.2,86.2


**market state at the start vs the end (Gold snapshot)**

In [0]:
snap = spark.table("workspace.gold.fact_game_daily_snapshot")

def kpis(key):
    return (snap.where(F.col("date_key") == key).agg(
        F.count("*").alias("games_tracked"),
        F.sum(F.col("is_on_sale").cast("int")).alias("games_on_sale"),
        F.round(F.avg(F.when(F.col("is_on_sale"), F.col("discount_pct"))), 1).alias("avg_sale_discount_pct"),
        F.sum((F.col("price_usd") == 0).cast("int")).alias("free_games"),
        F.round(F.avg(F.when(F.col("price_usd") > 0, F.col("price_usd"))), 2).alias("avg_paid_price_usd"),
        F.round(F.avg("approval_rate") * 100, 1).alias("avg_approval_pct"),
        F.sum("total_reviews").alias("total_reviews"),
        F.sum("owners_mid").alias("est_total_owners"))
      .first().asDict())

a, b = kpis(start_key), kpis(end_key)
rows = [(k, a[k], b[k], (b[k] - a[k]) if a[k] is not None and b[k] is not None else None) for k in a]
print(f"Market state: {start_eff} vs {end_eff}")
display(spark.createDataFrame(rows, "metric string, start double, end double, change double"))

Market state: 2026-10-09 vs 2026-10-25


metric,start,end,change
games_tracked,82040.0,82400.0,360.0
games_on_sale,32591.0,7584.0,-25007.0
avg_sale_discount_pct,59.0,64.1,5.099999999999994
free_games,10677.0,10729.0,52.0
avg_paid_price_usd,6.52,8.71,2.1900000000000013
avg_approval_pct,75.4,75.4,0.0
total_reviews,1.47733129E8,1.47834673E8,101544.0
est_total_owners,1.063676E10,1.065473E10,1.797E7


%md
### Price and discount activity
Every price or discount change between the two dates, from `fact_price_change`: counts per change type, the publishers that started the most sales, and the biggest price drops.

In [0]:
pc = spark.table("workspace.gold.fact_price_change")
lo = start_key if start_key == end_key else start_key + 0   # same day: that day only
in_range = pc.where((F.col("date_key") > start_key) & (F.col("date_key") <= end_key)) \
    if start_key != end_key else pc.where(F.col("date_key") == start_key)

types = spark.table("workspace.gold.dim_change_type")
pubs = spark.table("workspace.gold.dim_publisher")
games = spark.table("workspace.gold.dim_game")

print("Changes per type:")
display(in_range.join(types, "change_type_key").groupBy("change_type").agg(
    F.count("*").alias("events"),
    F.round(F.avg("new_discount_pct"), 1).alias("avg_new_discount_pct"))
  .orderBy(F.desc("events")))

print("Publishers that started the most sales:")
display(in_range.where("change_type_key = 1").join(pubs, "publisher_key")
  .where("publisher_key <> -1")
  .groupBy("publisher_name").agg(
      F.count("*").alias("sales_started"),
      F.round(F.avg("new_discount_pct"), 1).alias("avg_discount_pct"))
  .orderBy(F.desc("sales_started")).limit(10))

print("Biggest price drops:")
display(in_range.where("price_change_usd < 0 AND old_initialprice_usd < 100").join(games, "game_key")
  .select("game_name", "date_key", "old_price_usd", "new_price_usd", "price_change_usd", "new_discount_pct")
  .orderBy("price_change_usd").limit(10))

Changes per type:


change_type,events,avg_new_discount_pct
Sale ended,30177,0.0
Sale started,5120,64.5
Discount changed,600,61.0
Price increase,90,19.3
Price decrease,82,12.7


Publishers that started the most sales:


publisher_name,sales_started,avg_discount_pct
Conglomerate 5,116,84.4
My Way Games,71,80.5
Laush Studio,70,77.6
Daedalic Entertainment,66,85.7
Hede,50,95.0
Tero Lunkka,48,85.1
Focus Entertainment,44,81.2
Gamesforgames,44,81.8
GrabTheGames,44,77.0
Headup,40,84.5


Biggest price drops:


game_name,date_key,old_price_usd,new_price_usd,price_change_usd,new_discount_pct
The Ultimate Shot,20261023,99.99,4.99,-95.0,95.0
The Hidden Ghost,20261022,99.99,4.99,-95.0,95.0
Suicide Squad: Kill the Justice League,20261018,69.99,3.49,-66.5,95.0
Cardfight!! Vanguard Dear Days,20261019,69.99,4.89,-65.1,0.0
RPG Maker MV,20261015,79.99,15.99,-64.0,80.0
The First Thrust of God,20261013,59.99,2.99,-57.0,95.0
Aquarelle,20261024,59.99,2.99,-57.0,95.0
Crazy Car,20261015,59.99,2.99,-57.0,95.0
Back 4 Blood,20261017,59.99,5.99,-54.0,90.0
Hogwarts Legacy,20261017,59.99,5.99,-54.0,90.0


### New listings
Games that entered SteamSpy's list between the two dates, from `fact_game_listing`: totals by price tier and owners band, and the most played new games.|

In [0]:
lst = spark.table("workspace.gold.fact_game_listing")
in_range_l = lst.where((F.col("date_key") > start_key) & (F.col("date_key") <= end_key)) \
    if start_key != end_key else lst.where(F.col("date_key") == start_key)

print("New listings:", in_range_l.count())

print("By price tier:")
display(in_range_l.join(spark.table("workspace.gold.dim_price_tier"), "price_tier_key")
  .groupBy("tier_name", "sort_order").count().orderBy("sort_order").drop("sort_order"))

print("By owners band:")
display(in_range_l.join(spark.table("workspace.gold.dim_owners_band"), "owners_band_key")
  .groupBy("band_label", "sort_order").count().orderBy("sort_order").drop("sort_order"))

print("Most played new games:")
display(in_range_l.join(games, "game_key")
  .join(snap.where(F.col("date_key") == end_key).select("game_key", "ccu", "total_reviews", "price_usd"), "game_key")
  .join(pubs, "publisher_key")
  .select("game_name", "publisher_name", "ccu", "total_reviews", "price_usd", "date_key")
  .orderBy(F.desc("ccu"), F.desc("total_reviews")).limit(10))

New listings: 360
By price tier:


tier_name,count
Free,56
Under $5,128
$5 to $20,159
$20 to $40,15
Over $40,2


By owners band:


band_label,count
20k to 50k,252
50k to 100k,94
100k to 200k,14


Most played new games:


game_name,publisher_name,ccu,total_reviews,price_usd,date_key
Cauldron,Caketown Interactive,1682,500,14.99,20261024
Desktop Survivors 98,Brandon Hesslau,160,582,4.99,20261010
SWORN,Team17,150,2461,8.49,20261024
Keep Driving,YCJY Games,133,2770,12.59,20261017
Eden Crafters,Osaris Games,125,1417,9.99,20261011
Dolls Nest,NITRO PLUS,118,1274,19.99,20261024
Obenseuer,Loiste Interactive,101,1077,19.99,20261010
Ultimate General: American Revolution,Game-Labs,71,888,49.99,20261024
Promise Mascot Agency,Kaizen Game Works,54,1317,14.99,20261024
TRAIN CREW,Mizotsuki Rail / acty,38,337,16.99,20261024


### Publisher landscape on the end date
Top publishers on the end date by estimated owners, with game count, average approval and share of games on sale, compared with the start date.

In [0]:
def publisher_view(key):
    return (snap.where((F.col("date_key") == key) & (F.col("publisher_key") != -1))
            .groupBy("publisher_key").agg(
                F.count("*").alias("games"),
                F.sum("owners_mid").alias("est_owners"),
                F.round(F.avg("approval_rate") * 100, 1).alias("avg_approval_pct"),
                F.round(F.avg(F.col("is_on_sale").cast("int")) * 100, 1).alias("pct_on_sale")))

start_p = publisher_view(start_key).select(
    "publisher_key", F.col("games").alias("games_start"), F.col("pct_on_sale").alias("pct_on_sale_start"))
end_p = publisher_view(end_key)

print(f"Top 15 publishers on {end_eff} by estimated owners:")
display(end_p.join(start_p, "publisher_key", "left").join(pubs, "publisher_key")
  .select("publisher_name", "games",
          (F.col("games") - F.coalesce("games_start", F.lit(0))).alias("games_added"),
          "est_owners", "avg_approval_pct", "pct_on_sale", "pct_on_sale_start")
  .orderBy(F.desc("est_owners")).limit(15))

Top 15 publishers on 2026-10-25 by estimated owners:


publisher_name,games,games_added,est_owners,avg_approval_pct,pct_on_sale,pct_on_sale_start
Valve,35,0,568245000,89.2,0.0,40.0
Electronic Arts,138,0,366685000,79.3,0.0,48.6
Ubisoft,150,0,206590000,76.0,0.7,66.0
Rockstar Games,25,0,172085000,81.9,20.0,60.0
2K,99,0,168065000,72.7,18.2,51.5
Activision,69,0,159545000,79.3,1.4,39.1
KRAFTON,10,0,158485000,75.5,0.0,40.0
Amazon Games,3,0,153500000,60.9,0.0,0.0
Bethesda Softworks,65,0,150980000,80.7,10.8,70.8
SEGA,182,1,147925000,84.8,3.8,53.0
